In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image, ImageFile
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix


# Import required libraries and set up environment

ImageFile.LOAD_TRUNCATED_IMAGES = True

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

# Give a seed value to ensure reproducibility of results across runs.

# Check for GPU availability and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


In [ ]:
# Define constants and paths

MAG = "100X"
BASE_DIR = Path.cwd()
DATA_ROOT = BASE_DIR / "BreaKHis_v1" / "histology_slides" / "breast"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MAG     : {MAG}")

BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 0


BASE_DIR : c:\Users\sarpustacom\ASIL_Lab\Courses_Assignments\BreastCancerV1
DATA_ROOT: c:\Users\sarpustacom\ASIL_Lab\Courses_Assignments\BreastCancerV1\BreaKHis_v1\histology_slides\breast
MAG     : 100X


In [ ]:
# Define class names and mapping

CLASS_NAMES = [
    "adenosis",
    "fibroadenoma",
    "phyllodes_tumor",
    "tubular_adenoma",
    "ductal_carcinoma",
    "lobular_carcinoma",
    "mucinous_carcinoma",
    "papillary_carcinoma"
]

class_to_idx = {cls: i for i, cls in enumerate(CLASS_NAMES)}


In [ ]:
# Load dataset and prepare samples

def load_breakhis(magnification: str = "100X"):
    samples = []
    magnification = magnification.upper()

    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")

    for img_path in DATA_ROOT.rglob("*.png"):
        if img_path.parent.name.upper() != magnification:
            continue

        parts = set(img_path.parts)
        cls = next((name for name in CLASS_NAMES if name in parts), None)
        if cls is None:
            continue

        label = class_to_idx[cls]
        samples.append((str(img_path), label))

    return sorted(samples)


In [ ]:
# Define custom Dataset class

class BreakHisDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                if self.transform is not None:
                    img = self.transform(img)
            return img, torch.tensor(label, dtype=torch.long)
        except Exception as exc:
            raise RuntimeError(f"Failed to load image: {img_path}") from exc


In [ ]:
# Define data transformations

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((230, 230)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# Load dataset and split into train/val/test sets

full_samples = load_breakhis(MAG)
if not full_samples:
    raise ValueError(f"No samples found for magnification={MAG}. Check DATA_ROOT and folder names.")

labels = [s[1] for s in full_samples]
print(f"Total samples: {len(full_samples)}")
print("Full distribution:", Counter(labels))

# Stratified split to maintain class distribution across train/val/test sets

train_samples, temp_samples = train_test_split(
    full_samples,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.50,
    stratify=[s[1] for s in temp_samples],
    random_state=42,
)

# Print dataset statistics
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")
print("Train distribution:", Counter([s[1] for s in train_samples]))
print("Val distribution  :", Counter([s[1] for s in val_samples]))
print("Test distribution :", Counter([s[1] for s in test_samples]))


Total samples: 2081
Full distribution: Counter({4: 903, 1: 260, 6: 222, 5: 170, 3: 150, 7: 142, 2: 121, 0: 113})
Train: 1456 | Val: 312 | Test: 313
Train distribution: Counter({4: 632, 1: 182, 6: 155, 5: 119, 3: 105, 7: 99, 2: 85, 0: 79})
Val distribution  : Counter({4: 135, 1: 39, 6: 33, 5: 25, 3: 23, 7: 22, 2: 18, 0: 17})
Test distribution : Counter({4: 136, 1: 39, 6: 34, 5: 26, 3: 22, 7: 21, 2: 18, 0: 17})


In [ ]:
# Create Datasets and DataLoaders with weighted sampling to handle class imbalance
train_targets = [s[1] for s in train_samples]
class_sample_count = Counter(train_targets)
class_weights = torch.tensor(
    [1.0 / class_sample_count[i] for i in range(len(CLASS_NAMES))],
    dtype=torch.float,
)
sample_weights = torch.tensor([class_weights[t] for t in train_targets], dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)
# Create Datasets
train_ds = BreakHisDataset(train_samples, train_tf)
val_ds = BreakHisDataset(val_samples, test_tf)
test_ds = BreakHisDataset(test_samples, test_tf)


In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)


In [ ]:
# Create model, modify classifier for our number of classes, and move to device
model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(CLASS_NAMES))
model = model.to(device)

print(f"Classifier out_features: {model.classifier[2].out_features}")


Classifier out_features: 8


In [ ]:
# Define loss function, optimizer, and learning rate scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=LEARNING_RATE, 
    steps_per_epoch=len(train_loader), 
    epochs=NUM_EPOCHS,
    pct_start=0.3 
)

In [ ]:
# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return acc, p, r, f

best_f1 = 0.0
best_model_path = MODEL_DIR / f"best_convnext_tiny_{MAG.lower()}.pth"


In [ ]:
# Training loop with validation and model checkpointing
print(f"{'Epoch':^6} | {'Train Loss':^10} | {'Val Loss':^10} | {'Val Acc':^9} | {'Val Prec':^9} | {'Val Rec':^9} | {'Val F1':^9} | {'Status'}")
print("-" * 100)

# Iterate over epochs, training the model and evaluating on the validation set. Save the best model based on validation F1 score.
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    # Use tqdm to show progress bar for training loop
    # tqdm will automatically handle the progress bar display in Jupyter notebooks, showing the current epoch and progress through the training batches.
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
    # Calculate average training loss for the epoch
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    y_true = []
    y_pred = []

    # Evaluate on validation set without gradient computation to save memory and speed up inference
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc, val_prec, val_rec, val_f1 = calculate_metrics(y_true, y_pred)
    # Check if this is the best model based on validation F1 score and save it.
    status = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_f1": best_f1,
            "mag": MAG,
            "class_names": CLASS_NAMES,
        }, best_model_path)
        status = "⭐ BEST"

    # Print epoch results with metrics and status (indicating if it's the best model so far)
    print(
        f"{epoch:^6} | {avg_train_loss:^10.4f} | {avg_val_loss:^10.4f} | "
        f"{val_acc*100:^8.2f}% | {val_prec*100:^8.2f}% | {val_rec*100:^8.2f}% | {val_f1*100:^8.2f}% | {status}"
    )

# After training, print best validation F1 score and load the best model for final evaluation on the test set.
print(f"Best validation macro F1: {best_f1:.4f}")
print(f"Best model saved to: {best_model_path}")

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_loss = 0.0
y_true = []
y_pred = []

# Evaluate on test set using the best model

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(imgs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
test_acc, test_prec, test_rec, test_f1 = calculate_metrics(y_true, y_pred)

# Print final test results and classification report

print("TEST RESULTS")
print("-" * 60)
print(f"Test Loss      : {avg_test_loss:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Test Precision : {test_prec:.4f}")
print(f"Test Recall    : {test_rec:.4f}")
print(f"Test Macro F1  : {test_f1:.4f}")
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Epoch  | Train Loss |  Val Loss  |  Val Acc  | Val Prec  |  Val Rec  |  Val F1   | Status
----------------------------------------------------------------------------------------------------


Epoch 1:   0%|          | 0/46 [00:00<?, ?it/s]

  1    |   2.0674   |   2.0820   |  20.19  % |  22.54  % |  28.57  % |  20.81  % | ⭐ BEST


Epoch 2:   0%|          | 0/46 [00:00<?, ?it/s]

  2    |   1.8193   |   1.6196   |  54.49  % |  50.20  % |  59.91  % |  52.53  % | ⭐ BEST


...

  18   |   0.5043   |   0.6316   |  91.99  % |  93.40  % |  90.67  % |  91.96  % | 


Epoch 19:   0%|          | 0/46 [00:00<?, ?it/s]

  19   |   0.5090   |   0.6342   |  92.31  % |  93.64  % |  90.81  % |  92.14  % | 


Epoch 20:   0%|          | 0/46 [00:00<?, ?it/s]

  20   |   0.4995   |   0.6343   |  91.99  % |  93.06  % |  90.72  % |  91.81  % | 
Best validation macro F1: 0.9214
Best model saved to: c:\Users\sarpustacom\ASIL_Lab\Courses_Assignments\BreastCancerV1\models\best_convnext_tiny_100x.pth
TEST RESULTS
------------------------------------------------------------
Test Loss      : 0.6228
Test Accuracy  : 0.9169
Test Precision : 0.9137
Test Recall    : 0.9068
Test Macro F1  : 0.9090
Classification Report:
                     precision    recall  f1-score   support

           adenosis       0.94      0.88      0.91        17
       fibroadenoma       0.95      1.00      0.97        39
    phyllodes_tumor       1.00      0.89      0.94        18
    tubular_adenoma       0.96      1.00      0.98        22
   ductal_carcinoma       0.93      0.93      0.93       136
  lobular_carcinoma       0.78      0.69      0.73        26
 mucinous_carcinoma       0.89      0.91      0.90        34
papillary_carcinoma       0.87      0.95      0.91      